# Sentiment Analysis of Amazon Reviews Using Deep Learning
## IE 7265 - Deep Learning for AI
### Wiktoria Lasek

---

## Project Overview
This project builds a progressive deep learning pipeline for sentiment analysis 
of Amazon customer reviews. Four models are compared with attention visualization for interpretability and cross-domain testing on Yelp reviews.

## Datasets
- **Amazon Reviews** - 3.6M reviews (train: 200,000 sample, test: 40,000 sample)
- **Yelp Reviews** - 38,000 reviews (5,000 sample for cross-domain testing)

## Models
1. Logistic Regression (baseline)
2. Feedforward Neural Network
3. LSTM with GloVe embeddings
4. DistilBERT (fine-tuned transformer)

## Section 1: Setup
### Importing required libraries for data loading, processing and visualization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Libraries loaded:
- **Pandas** - data loading and manipulation
- **NumPy** - numerical operations
- **Matplotlib** - visualizations

## Section 2: Data Loading and Exploration

### 2.1 Amazon Reviews Dataset
The Amazon Reviews dataset is loaded from a .txt file where each line contains 
a label (__label__1 or __label__2) followed by the review text.
The file is read line by line and split into two columns - label and review.

In [2]:
reviews = []
labels = []

with open('/Users/wiktorialasek/Downloads/archive-2/train.ft.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        parts = line.split(' ', 1)
        if len(parts) == 2:
            labels.append(parts[0])
            reviews.append(parts[1])

train_df = pd.DataFrame({'label': labels, 'review': reviews})
print(f"Train data shape: {train_df.shape}")
train_df.head()

Train data shape: (3600000, 2)


,label,review
0,__label__2,Stuning even for the non-gamer: This sound tra...
1,__label__2,The best soundtrack ever to anything.: I'm rea...
2,__label__2,Amazing!: This soundtrack is my favorite music...
3,__label__2,Excellent Soundtrack: I truly like this soundt...
4,__label__2,"Remember, Pull Your Jaw Off The Floor After He..."


### Findings:
- Successfully loaded 3,600,000 Amazon reviews
- Two columns: label and review text
- Labels: __label__1 (negative), __label__2 (positive)

In [3]:
print("Missing values:")
print(train_df.isnull().sum())
print(f"Total missing: {train_df.isnull().sum().sum()}")

Missing values:
label     0
review    0
dtype: int64
Total missing: 0


### Findings:
- Zero missing values in both label and review columns
- Dataset is completely clean - no imputation needed

In [4]:
print("Label distribution:")
print(train_df['label'].value_counts())

Label distribution:
label
__label__2    1800000
__label__1    1800000
Name: count, dtype: int64


### Findings:
- Perfectly balanced dataset - 1,800,000 positive and 1,800,000 negative reviews
- 50/50 class balance - ideal for binary classification
- No class imbalance issues 

In [5]:
print("POSITIVE review example:")
print(train_df[train_df['label']=='__label__2']['review'].iloc[0])
print()
print("NEGATIVE review example:")
print(train_df[train_df['label']=='__label__1']['review'].iloc[0])

POSITIVE review example:
Stuning even for the non-gamer: This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^

NEGATIVE review example:
Buyer beware: This is a self-published book, and if you want to know why--read a few paragraphs! Those 5 star reviews must have been written by Ms. Haddon's family and friends--or perhaps, by herself! I can't imagine anyone reading the whole thing--I spent an evening with the book and a friend and we were in hysterics reading bits and pieces of it to one another. It is most definitely bad enough to be entered into some kind of a "worst book" contest. I can't believe Amazon even sells this kind of thing. Maybe I can o

### 2.2 Sampling Strategy
Due to computational constraints, stratified sample is taken from the full dataset.
A random sample of 200,000 training reviews is selected - maintaining the original 50/50 class balance.

In [7]:
train_sample = train_df.sample(n=200000, random_state=42)
print(f"Sample shape: {train_sample.shape}")

Sample shape: (200000, 2)


In [30]:
train_sample['review_length'] = train_sample['review'].apply(lambda x: len(x.split()))
print("Amazon Train review length:")
print(train_sample['review_length'].describe())

Amazon Train review length:
count    200000.000000
mean         78.481315
std          42.860370
min          10.000000
25%          42.000000
50%          70.000000
75%         108.000000
max         213.000000
Name: review_length, dtype: float64


### 2.3 Amazon Reviews - Test Dataset
The test dataset is loaded using the same method as the training data. 
It is used exclusively for model evaluation - never seen during training.


In [8]:
reviews_test = []
labels_test = []

with open('/Users/wiktorialasek/Downloads/archive-2/test.ft.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        parts = line.split(' ', 1)
        if len(parts) == 2:
            labels_test.append(parts[0])
            reviews_test.append(parts[1])

test_df = pd.DataFrame({'label': labels_test, 'review': reviews_test})
print(f"Test data shape: {test_df.shape}")

Test data shape: (400000, 2)


In [32]:
test_df.head()

,label,review
0,__label__2,Great CD: My lovely Pat has one of the GREAT v...
1,__label__2,One of the best game music soundtracks - for a...
2,__label__1,Batteries died within a year ...: I bought thi...
3,__label__2,"works fine, but Maha Energy is better: Check o..."
4,__label__2,Great for the non-audiophile: Reviewed quite a...


### Findings:
- Successfully loaded 400,000 Amazon test reviews
- Two columns: label and review text 

In [10]:
print("Label distribution:")
print(test_df['label'].value_counts())

Label distribution:
label
__label__2    200000
__label__1    200000
Name: count, dtype: int64


### Findings:
- Perfectly balanced - 200,000 positive, 200,000 negative
- 50/50 class balance maintained 

In [9]:
print("Missing values:")
print(test_df.isnull().sum())
print(f"Total missing: {test_df.isnull().sum().sum()}")

Missing values:
label     0
review    0
dtype: int64
Total missing: 0


### Findings:
- Zero missing values in both columns
- Dataset is completely clean 

In [11]:
test_sample = test_df.sample(n=40000, random_state=42)
print(f"Test sample shape: {test_sample.shape}")

Test sample shape: (40000, 2)


In [33]:
print("Label distribution:")
print(test_sample['label'].value_counts())

Label distribution:
label
1    20009
0    19991
Name: count, dtype: int64


In [31]:
test_sample['review_length'] = test_sample['review'].apply(lambda x: len(x.split()))
print("\nTest review length:")
print(test_sample['review_length'].describe())


Test review length:
count    40000.000000
mean        78.486750
std         42.844235
min          8.000000
25%         42.000000
50%         70.000000
75%        108.000000
max        218.000000
Name: review_length, dtype: float64


### Findings:
- Test sample: 40,000 reviews (20,009 positive, 19,991 negative) 
- Mean review length: 78 words, median: 70 words
- Similar length distribution to training sample 
- Sample maintains class balance and representative review lengths

## Label Conversion
Amazon labels are converted from __label__1/__label__2 to binary 0/1:
- __label__1 - 0 (negative)
- __label__2 - 1 (positive)


In [13]:
#Convert Amazon labels to 0 and 1
train_sample['label'] = train_sample['label'].map({'__label__1': 0, '__label__2': 1})
test_sample['label'] = test_sample['label'].map({'__label__1': 0, '__label__2': 1})

print("Train labels after conversion:")
print(train_sample['label'].value_counts())
print("\nTest labels after conversion:")
print(test_sample['label'].value_counts())

Train labels after conversion:
label
1    100020
0     99980
Name: count, dtype: int64

Test labels after conversion:
label
1    20009
0    19991
Name: count, dtype: int64


## 2.4 Yelp Reviews Dataset (Cross-Domain Test)
The Yelp dataset contains restaurant and business reviews.
It is used exclusively for cross-domain generalization testing.
The model is never trained on this data.

In [25]:
yelp_df = pd.read_csv('/Users/wiktorialasek/Downloads/archive-3/test.csv')
yelp_df.shape

(38000, 2)

In [26]:
yelp_df.head()

,text,label
0,"Contrary to other reviews, I have zero complai...",1
1,Last summer I had an appointment to get new ti...,0
2,"Friendly staff, same starbucks fair you get an...",1
3,The food is good. Unfortunately the service is...,0
4,Even when we didn't have a car Filene's Baseme...,1


### Findings:
- Successfully loaded 38,000 Yelp reviews
- Two columns: text, label (0=negative, 1=positive) 

In [28]:
print("Yelp missing values:")
print(yelp_sample.isnull().sum())

Yelp missing values:
text            0
label           0
clean_review    0
dtype: int64


### Findings:
- Zero missing values 
- Dataset is completely clean

In [ ]:
yelp_sample = yelp_df.sample(n=5000, random_state=42)

print(f"Yelp sample: {yelp_sample.shape}")
print(yelp_sample['label'].value_counts())

In [ ]:
print("Label distribution:")
print(test_sample['label'].value_counts())

In [29]:
yelp_sample['review_length'] = yelp_sample['text'].apply(lambda x: len(x.split()))
print("\nYelp review length:")
print(yelp_sample['review_length'].describe())


Yelp review length:
count    5000.000000
mean      133.371400
std       121.649237
min         1.000000
25%        52.000000
50%        97.000000
75%       175.000000
max       988.000000
Name: review_length, dtype: float64


### Findings:
- Yelp sample: 5,000 reviews (2,540 negative, 2,460 positive) 
- Mean review length: 133 words, median: 97 words
- Notably longer than Amazon reviews (mean 78 words)
- This difference in writing style makes Yelp a meaningful cross-domain test 

## Section 3: Text Preprocessing & Feature Engineering

### What was done:
- Applied heavy text preprocessing for Logistic Regression and Feedforward Neural Network:
  - Converted all text to lowercase
  - Removed punctuation and special characters
  - Removed stop words (e.g. "the", "a", "is", "was")
- Applied TF-IDF vectorization with max_features=10,000
  - Converts each review into a numerical vector of 10,000 word importance scores
  - Fitted on training data only - test and Yelp data transformed using the same vocabulary

### Results:
| Dataset | Shape after TF-IDF |
|---|---|
| Amazon Train | (200,000 × 10,000) |
| Amazon Test | (40,000 × 10,000) |
| Yelp (cross-domain) | (5,000 × 10,000) |

---

In [14]:
# Install nltk for stop words
import nltk
nltk.download('stopwords')
nltk.download('punkt')
print("NLTK downloaded successfully!")

NLTK downloaded successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/wiktorialasek/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/wiktorialasek/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [15]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = text.strip()
    # Remove stop words
    words = text.split()
    words = [w for w in words if w not in stop_words]
    # Join back to string
    text = ' '.join(words)
    return text

#Apply to train and test
print("Cleaning train data...")
train_sample['clean_review'] = train_sample['review'].apply(clean_text)

print("Cleaning test data...")
test_sample['clean_review'] = test_sample['review'].apply(clean_text)

print("Done!")
print("\nOriginal review:")
print(train_sample['review'].iloc[0])
print("\nCleaned review:")
print(train_sample['clean_review'].iloc[0])

Cleaning train data...
Cleaning test data...
Done!

Original review:
Expensive Junk: This product consists of a piece of thin flexible insulating material, adhesive backed velcro and white electrical tape.Problems:1. Instructions are three pictures with little more information.2. Velcro was all crumpled as received and was stronger than the adhesive. When i tried to disengage the velcro both pieces came off and the paint from the ceiling.3. White electrical tape was horrible... cheap, narrow and it fell off in less than 1 hour.4. The price is a ripoff.I am building my own which is easier to use, cheaper, more attractive, and higher r-value. I am surprised Amazon even lists this junk.

Cleaned review:
expensive junk product consists piece thin flexible insulating material adhesive backed velcro white electrical tapeproblems instructions three pictures little information velcro crumpled received stronger adhesive tried disengage velcro pieces came paint ceiling white electrical tape horr

In [16]:
# Clean Yelp data
print("Cleaning Yelp data...")
yelp_sample['clean_review'] = yelp_sample['text'].apply(clean_text)

print("Done!")
print("\nOriginal Yelp review:")
print(yelp_sample['text'].iloc[0])
print("\nCleaned Yelp review:")
print(yelp_sample['clean_review'].iloc[0])

Cleaning Yelp data...
Done!

Original Yelp review:
I expected the prices of the entrees to be a little bit higher but the quality of the Chinese food was not worth the money I paid for the dishes. I got the 18 monk noodle and the traditional dimsum. If I could describe the food  in one word-terrible! Making the dimsum look pretty by topping it with gold flakes did not do anything to make up for the flavor of the dimsum. It  seemed too starchy and you can hardly taste the meat. The noodles looked like a sad , greasy slop of Mai fun type noodles (noodles were stuck together) saturated with soy sauce for color, and garnished with a few pieces of shitake mushrooms, green onions and fine threads of carrots. And yes, portions were small, but that's not really the worst part of the whole experience. Just poorly prepared, way overpriced Chinese food...sorry.

Cleaned Yelp review:
expected prices entrees little bit higher quality chinese food worth money paid dishes got monk noodle traditional 

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

#Create TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=10000)

#Fit on training data and transform
print("Applying TF-IDF...")
X_train = tfidf.fit_transform(train_sample['clean_review'])
X_test = tfidf.transform(test_sample['clean_review'])
X_yelp = tfidf.transform(yelp_sample['clean_review'])

#Labels
y_train = train_sample['label']
y_test = test_sample['label']
y_yelp = yelp_sample['label']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"X_yelp shape: {X_yelp.shape}")

Applying TF-IDF...
X_train shape: (200000, 10000)
X_test shape: (40000, 10000)
X_yelp shape: (5000, 10000)


## Section 4: Model 1 - Logistic Regression (Baseline)

## 4.1 Logistic Regression - Amazon Train and Test Sets


### What was done:
- Trained Logistic Regression on 200,000 Amazon training reviews
- Evaluated on 40,000 Amazon test reviews
- Extracted top 10 most positive and negative words for interpretability analysis



In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

#Train Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

#Evaluate on Amazon test set
lr_preds = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds)

print(f"\nLOGISTIC REGRESSION RESULTS")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"F1-Score: {lr_f1:.4f}")
print("\nDetailed Report:")
print(classification_report(y_test, lr_preds))

Training Logistic Regression...

LOGISTIC REGRESSION RESULTS
Accuracy: 0.8882
F1-Score: 0.8886

Detailed Report:
              precision    recall  f1-score   support

           0       0.89      0.88      0.89     19991
           1       0.89      0.89      0.89     20009

    accuracy                           0.89     40000
   macro avg       0.89      0.89      0.89     40000
weighted avg       0.89      0.89      0.89     40000



In [19]:
import numpy as np

#Get feature names
feature_names = tfidf.get_feature_names_out()

#Get LR coefficients
coefficients = lr_model.coef_[0]

#Top 10 most positive words
top_positive_idx = np.argsort(coefficients)[-10:][::-1]
top_negative_idx = np.argsort(coefficients)[:10]

print("TOP 10 POSITIVE WORDS")
for idx in top_positive_idx:
    print(f"{feature_names[idx]}: {coefficients[idx]:.4f}")

print("\nTOP 10 NEGATIVE WORDS")
for idx in top_negative_idx:
    print(f"{feature_names[idx]}: {coefficients[idx]:.4f}")

TOP 10 POSITIVE WORDS
great: 11.7152
excellent: 11.1935
perfect: 8.2668
awesome: 7.9586
best: 7.7696
fantastic: 7.6372
amazing: 7.4196
love: 7.2617
highly: 7.1426
wonderful: 7.0742

TOP 10 NEGATIVE WORDS
worst: -13.1107
disappointing: -12.9731
waste: -12.0526
disappointment: -10.8487
disappointed: -10.5225
boring: -9.7860
poor: -9.6521
poorly: -9.3829
terrible: -8.9727
horrible: -8.7327


### Results on Amazon Test Set:
| Metric | Score |
|---|---|
| Accuracy | 88.82% |
| F1-Score | 88.86% |
| Precision | 89% |
| Recall | 89% |

### Interpretability - Top Words:

**Most Positive Words:**
great (11.72), excellent (11.19), perfect (8.27), awesome (7.96), best (7.77),
fantastic (7.64), amazing (7.42), love (7.26), highly (7.14), wonderful (7.07)

**Most Negative Words:**
worst (-13.11), disappointing (-12.97), waste (-12.05), disappointment (-10.85),
disappointed (-10.52), boring (-9.79), poor (-9.65), poorly (-9.38),
terrible (-8.97), horrible (-8.73)

### Key Conclusions:
- Logistic Regression achieves a strong baseline of 88.82% accuracy using only word frequencies
- The model correctly learned intuitive sentiment words - positive words like "great" and "excellent",
  negative words like "worst" and "disappointing"
- This gives us a transparent, human-readable explanation of model decisions
- This is our benchmark score - LSTM and DistilBERT must beat this
- The high baseline suggests Amazon reviews contain clear, unambiguous sentiment language

## 4.2 Logistic Regression - Cross-Domain Test (Yelp)

The trained Logistic Regression model is tested on 5,000 Yelp reviews - 
data it was never trained on and from a completely different domain 
(restaurant/business reviews vs Amazon product reviews).



In [20]:
# Test LR on Yelp
lr_yelp_preds = lr_model.predict(X_yelp)
lr_yelp_accuracy = accuracy_score(y_yelp, lr_yelp_preds)
lr_yelp_f1 = f1_score(y_yelp, lr_yelp_preds)

print("LR CROSS-DOMAIN TEST (Yelp)")
print(f"Accuracy: {lr_yelp_accuracy:.4f}")
print(f"F1-Score: {lr_yelp_f1:.4f}")
print("\nDetailed Report:")
print(classification_report(y_yelp, lr_yelp_preds))

LR CROSS-DOMAIN TEST (Yelp)
Accuracy: 0.8786
F1-Score: 0.8826

Detailed Report:
              precision    recall  f1-score   support

           0       0.92      0.83      0.87      2540
           1       0.84      0.93      0.88      2460

    accuracy                           0.88      5000
   macro avg       0.88      0.88      0.88      5000
weighted avg       0.88      0.88      0.88      5000



### Results:
| Metric | Score |
|---|---|
| Accuracy | 87.86% |
| F1-Score | 88.26% |

### Key Conclusions:
- Only 1% drop in accuracy compared to Amazon test (88.82% -> 87.86%) ✅
- Model generalizes well to a completely different review domain
- Suggests Logistic Regression learned genuine sentiment patterns
- not just Amazon-specific vocabulary
- Strong baseline for cross-domain performance - LSTM and DistilBERT must beat this